# Baran Corrector
Melakukan corrector menggunakan output dari Raha dan DAE (detector)

### Import Library

In [1]:
import pickle
import time
import raha
import yaml

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

### Configruation

In [2]:
# CONFIGURATION

BARAN_LABELING_BUDGET = 2
BARAN_VERBOSE = False
BARAN_AUTO_SAVE = True

# DICTIONARIES_YAML_PATH = "./dicts_minimal.yaml"
DICTIONARIES_YAML_PATH = "./dicts_full.yaml"

# RAHA
RAHA_DETECTION_RESULTS_PATH = "./results/detection/raha"
RAHA_CORRECTION_RESULTS_PATH = "./results/correction/raha"

# DAE
DAE_DETECTION_RESULTS_PATH = "./results/detection/DAE"
DAE_CORRECTION_RESULTS_PATH = "./results/correction/DAE"

extension = ".pkl"

### Load YAML

In [3]:
yaml_path = "dicts_full.yaml"

with open(yaml_path, "rb") as file:
  dictionaries = yaml.safe_load(file)

### Raha Correction using Baran

In [4]:
raha_correction = raha.Correction()
raha_correction.LABELING_BUDGET = BARAN_LABELING_BUDGET
raha_correction.VERBOSE = BARAN_VERBOSE
raha_correction.SAVE_RESULTS = BARAN_AUTO_SAVE

for name, item in dictionaries.items():
  start_time = time.time()
  print(f'Running correction on {name} from RAHA')

  # Load Raha Result
  with open(f"{RAHA_DETECTION_RESULTS_PATH}/{name}{extension}", "rb") as f:
    d = pickle.load(f)
    print(f"Pickle {RAHA_DETECTION_RESULTS_PATH}/{name} loaded")

  raha_correction.initialize_models(d)
  raha_correction.initialize_dataset(d)

  for si in d.labeled_tuples:
    d.sampled_tuple = si
    raha_correction.update_models(d)
    raha_correction.predict_corrections(d)

  # save correction
  with open(f"{RAHA_CORRECTION_RESULTS_PATH}/{name}.pkl", "wb") as f:
    pickle.dump(d, f)
    print(f"Pickle raha correction saved to {RAHA_CORRECTION_RESULTS_PATH}/{name}.pkl")

  total_time = time.time() - start_time
  print(f"total time on {name}: {total_time:.1f} seconds")
  print(f"-" * 50)

Running correction on beers from RAHA
Pickle ./results/detection/raha/beers loaded
Pickle raha correction saved to ./results/correction/raha/beers.pkl
total time on beers: 308.1 seconds
--------------------------------------------------
Running correction on bike from RAHA
Pickle ./results/detection/raha/bike loaded
Pickle raha correction saved to ./results/correction/raha/bike.pkl
total time on bike: 1127.4 seconds
--------------------------------------------------
Running correction on nasa from RAHA
Pickle ./results/detection/raha/nasa loaded
Pickle raha correction saved to ./results/correction/raha/nasa.pkl
total time on nasa: 269.8 seconds
--------------------------------------------------


### DAE Correction using Baran

In [5]:
dae_correction = raha.Correction()
dae_correction.LABELING_BUDGET = BARAN_LABELING_BUDGET
dae_correction.VERBOSE = BARAN_VERBOSE
dae_correction.SAVE_RESULTS = BARAN_AUTO_SAVE

for name, item in dictionaries.items():
  start_time = time.time()
  print(f'Running correction on {name} from DAE')

  # Load DAE Result
  with open(f"{DAE_DETECTION_RESULTS_PATH}/{name}{extension}", "rb") as f:
    d = pickle.load(f)
    print(f"Pickle {DAE_DETECTION_RESULTS_PATH}/{name} loaded")

  dae_correction.initialize_models(d)
  dae_correction.initialize_dataset(d)

  while len(d.labeled_tuples) < dae_correction.LABELING_BUDGET:
    dae_correction.sample_tuple(d)
    if d.has_ground_truth:
      dae_correction.label_with_ground_truth(d)
      dae_correction.update_models(d)
      dae_correction.predict_corrections(d)

  # save correction
  with open(f"{DAE_CORRECTION_RESULTS_PATH}/{name}.pkl", "wb") as f:
    pickle.dump(d, f)
    print(f"Pickle DAE correction saved to {DAE_CORRECTION_RESULTS_PATH}/{name}.pkl")

  total_time = time.time() - start_time
  print(f"total time on {name}: {total_time:.1f} seconds")
  print(f"-" * 50)

Running correction on beers from DAE
Pickle ./results/detection/DAE/beers loaded
Pickle DAE correction saved to ./results/correction/DAE/beers.pkl
total time on beers: 350.4 seconds
--------------------------------------------------
Running correction on bike from DAE
Pickle ./results/detection/DAE/bike loaded
Pickle DAE correction saved to ./results/correction/DAE/bike.pkl
total time on bike: 397.3 seconds
--------------------------------------------------
Running correction on nasa from DAE
Pickle ./results/detection/DAE/nasa loaded
Pickle DAE correction saved to ./results/correction/DAE/nasa.pkl
total time on nasa: 248.5 seconds
--------------------------------------------------
